# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MA-1305/Project_Mahin-1305/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/MA-1305/Project_Mahin-1305.git /content/Project_Mahin-1305

fatal: destination path '/content/Project_Mahin-1305' already exists and is not an empty directory.


In [19]:
from pathlib import Path

data_path = Path("/content/Project_Mahin-1305/data/raw/content_refresh_anonymized.csv")

print("File exists:", data_path.exists())
print("Path:", data_path)

File exists: True
Path: /content/Project_Mahin-1305/data/raw/content_refresh_anonymized.csv


In [20]:
import pandas as pd
import numpy as np

data_path = "/content/Project_Mahin-1305/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining_label"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Feature matrix shape: (30000, 27)
Target shape: (30000,)
Numeric features: 18
Categorical features: 9


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The feature vector contains numeric and categorical page-level signals.

Numeric features include search volume, competition, content length, impressions, clicks, sessions, content age, freshness, CTR, average position, engagement, scroll rate, and AI traffic.

Categorical features include competition level, content type, intent, age tier, freshness tier, word-count tier, character-count tier, impression tier, and position tier.

Missing numeric values will be handled with median filling, while missing categorical values will be handled with the most frequent category. These features are intended to be available before the prediction decision.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [22]:
display(df.dtypes)

,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked for fields that directly define or describe the target. trend_direction is used to create the is_declining_label, so it must not be used as a feature. trend_pct is also directly related to the observed trend outcome, so I excluded it.

avg_position and position_tier are not excluded just because their names contain "position"; they are observable search signals.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Columns excluded because they are target-derived or leakage-prone:")

leakage_columns = [
    "trend_direction",
    "trend_pct"
]

for col in leakage_columns:
    if col in df.columns:
        print("-", col)

Columns excluded because they are target-derived or leakage-prone:
- trend_direction
- trend_pct


In [24]:
print("All columns:")
for i, col in enumerate(df.columns):
    print(i, col)

All columns:
0 content_id
1 client_id
2 search_volume
3 competition
4 competition_level
5 cpc
6 content_type
7 main_intent
8 word_count
9 char_count
10 provider_used
11 model_used
12 impressions_90d
13 clicks_90d
14 pageviews_90d
15 sessions_90d
16 users_90d
17 engaged_sessions_90d
18 ai_sessions_90d
19 scroll_events_90d
20 days_with_impressions
21 days_with_sessions
22 impressions_last_30d
23 clicks_last_30d
24 sessions_last_30d
25 impressions_prev_30d
26 clicks_prev_30d
27 sessions_prev_30d
28 content_age_days
29 age_tier
30 age_tier_order
31 days_since_last_update
32 freshness_tier
33 word_count_tier
34 char_count_tier
35 ctr
36 avg_position
37 engagement_rate
38 scroll_rate
39 ai_traffic_pct
40 impression_tier
41 position_tier
42 trend_direction
43 trend_pct
44 is_declining_label


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded fields:

- content_id — identifier only; it is not a useful predictive feature.
- client_id — used for grouped validation, not as a predictive feature.
- provider_used — describes the data-generation process rather than the page opportunity.
- model_used — describes the data-generation process rather than the page opportunity.
- trend_direction — directly used to create the target, so using it would leak the answer.
- trend_pct — directly describes the trend outcome and is excluded.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct"
]

print("Excluded fields:")
for col in excluded_fields:
    print("-", col)

Excluded fields:
- content_id
- client_id
- provider_used
- model_used
- trend_direction
- trend_pct


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.